# Toy Loss Surface Comparison — Beale & Rosenbrock

All six optimisers race from the same start on two classic non-convex surfaces.
Watch how velocity, adaptation, and look-ahead shape each trajectory.

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import matplotlib.pyplot as plt
import numpy as np
from core.nn import (
    NAG,
    SGD,
    AdaGrad,
    Adam,
    Momentum,
    RMSProp,
)
from core.nn.parameter import Parameter

In [ ]:
# =====================================================================
# 1. Test functions with analytical gradients
# =====================================================================


def beale(xy):
    x, y = xy[0], xy[1]
    t1 = 1.5 - x + x * y
    t2 = 2.25 - x + x * y**2
    t3 = 2.625 - x + x * y**3
    return t1**2 + t2**2 + t3**2


def grad_beale(xy):
    x, y = xy[0], xy[1]
    t1 = 1.5 - x + x * y
    t2 = 2.25 - x + x * y**2
    t3 = 2.625 - x + x * y**3
    dx = 2 * t1 * (-1 + y) + 2 * t2 * (-1 + y**2) + 2 * t3 * (-1 + y**3)
    dy = 2 * t1 * x + 2 * t2 * 2 * x * y + 2 * t3 * 3 * x * y**2
    return np.array([dx, dy])


def rosenbrock(xy, a=1.0, b=100.0):
    x, y = xy[0], xy[1]
    return (a - x) ** 2 + b * (y - x**2) ** 2


def grad_rosenbrock(xy, a=1.0, b=100.0):
    x, y = xy[0], xy[1]
    dx = -2 * (a - x) - 4 * b * x * (y - x**2)
    dy = 2 * b * (y - x**2)
    return np.array([dx, dy])

In [ ]:
# =====================================================================
# 2. Optimiser runner — track trajectory + loss per step
# =====================================================================


def run_optimizer(opt_class, opt_kwargs, start, func, grad_func, n_steps=200):
    """Run an optimiser on an analytical test function.

    Wraps a single Parameter tracking (x, y); feeds analytical gradients
    into param.grad at each step.  Returns full trajectory and loss curve.
    """
    param = Parameter(start.copy())
    opt = opt_class([param], **opt_kwargs)

    trajectory = [start.copy()]
    losses = [func(start)]

    for _ in range(n_steps):
        param.grad = grad_func(param.data)
        opt.step()
        trajectory.append(param.data.copy())
        losses.append(func(param.data))

    return np.array(trajectory), np.array(losses), losses[-1] < 1e-6

In [ ]:
# =====================================================================
# 3. Per-optimiser hyper-parameters (tuned independently per problem)
# =====================================================================

np.random.seed(42)

BEALE_CONFIG = [
    ("SGD", SGD, {"lr": 0.03}),
    ("Momentum", Momentum, {"lr": 0.01, "momentum": 0.9}),
    ("NAG", NAG, {"lr": 0.01, "momentum": 0.9}),
    ("AdaGrad", AdaGrad, {"lr": 0.5}),
    ("RMSProp", RMSProp, {"lr": 0.01, "beta": 0.9}),
    ("Adam", Adam, {"lr": 0.20, "betas": (0.9, 0.999)}),
]

ROSEN_CONFIG = [
    ("SGD", SGD, {"lr": 0.003}),
    ("Momentum", Momentum, {"lr": 0.003, "momentum": 0.9}),
    ("NAG", NAG, {"lr": 0.001, "momentum": 0.9}),
    ("AdaGrad", AdaGrad, {"lr": 0.3}),
    ("RMSProp", RMSProp, {"lr": 0.003, "beta": 0.9}),
    ("Adam", Adam, {"lr": 0.50, "betas": (0.9, 0.999)}),
]

BEALE_STEPS = 500
ROSEN_STEPS = 1000

PROBLEMS = [
    {
        "name": "Beale",
        "func": beale,
        "grad": grad_beale,
        "configs": BEALE_CONFIG,
        "n_steps": BEALE_STEPS,
        "start": np.array([-0.5, 0.0]),
        "x_lim": (-1.0, 4.0),
        "y_lim": (-0.5, 1.5),
        "true_min": np.array([3.0, 0.5]),
    },
    {
        "name": "Rosenbrock",
        "func": rosenbrock,
        "grad": grad_rosenbrock,
        "configs": ROSEN_CONFIG,
        "n_steps": ROSEN_STEPS,
        "start": np.array([-1.2, 1.0]),
        "x_lim": (-1.5, 1.8),
        "y_lim": (-0.5, 2.0),
        "true_min": np.array([1.0, 1.0]),
    },
]

In [ ]:
# =====================================================================
# 4. Run all optimisers on all problems
# =====================================================================

results = {}

for problem in PROBLEMS:
    name = problem["name"]
    results[name] = {}
    print(f"\n{'=' * 55}")
    print(f"  {name} function — starting from {list(problem['start'])}")
    print(f"{'=' * 55}")
    for label, opt_class, kwargs in problem["configs"]:
        traj, losses, converged = run_optimizer(
            opt_class,
            kwargs,
            problem["start"],
            problem["func"],
            problem["grad"],
            problem["n_steps"],
        )
        results[name][label] = (traj, losses, converged)
        status = "✓" if converged else "✗"
        print(f"  {label:>10}: final loss {losses[-1]:.2e}  {status}")

### Why Adam needs a higher LR on 2D test functions

Adam uses `1/√v̂` to scale each step.  When initial gradients are large,
√v̂ becomes large too, suppressing the effective step dramatically:

| Problem | Initial ‖g‖ | √v̂(t=1) | Adam LR | Effective step | Momentum LR | Effective step |
|:-------:|:-----------:|:--------:|:-------:|:--------------:|:-----------:|:--------------:|
| Beale   | ≈ 16        | 0.50     | 0.20 →  | **0.40** ✅    | 0.01 →      | 0.16           |
| Rosenbrock | ≈ 233    | 7.36     | 0.50 →  | **0.07** ✅    | 0.003 →     | 0.70           |

That's why Adam at lr=0.01 on Beale (8× suppressed) and lr=0.02 on Rosenbrock
(233× suppressed) both looked terrible.  Raising the LR compensates.

In deep learning, BatchNorm and Xavier init keep gradients well-scaled,
so Adam's default lr=0.001 works out of the box.  These 2D functions have
no such normalisation, exposing the bare mechanics.

In [ ]:
# =====================================================================
# 5. Contour-plot helper
# =====================================================================

COLOURS = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628"]


def _contour_base(problem, ax):
    x_lim, y_lim = problem["x_lim"], problem["y_lim"]
    xs = np.linspace(x_lim[0], x_lim[1], 250)
    ys = np.linspace(y_lim[0], y_lim[1], 250)
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X)
    for i in range(len(xs)):
        for j in range(len(ys)):
            Z[j, i] = problem["func"](np.array([X[j, i], Y[j, i]]))

    if problem["name"] == "Rosenbrock":
        ax.contourf(X, Y, Z, levels=np.logspace(-2, 3, 30), cmap="bone_r", alpha=0.65)
    else:
        ax.contourf(X, Y, np.clip(Z, 0, 60), levels=25, cmap="bone_r", alpha=0.65)
    ax.contour(
        X, Y, np.clip(Z, 0, 200), levels=6, colors="gray", linewidths=0.5, alpha=0.4
    )
    ax.scatter(
        problem["true_min"][0],
        problem["true_min"][1],
        marker="*",
        s=200,
        c="gold",
        edgecolors="k",
        zorder=10,
        label="global min",
    )
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")


def make_contour_plot(problem, results_dict, ax):
    """Contour background + all optimiser trajectory overlays."""
    _contour_base(problem, ax)
    for idx, (label, _, _) in enumerate(problem["configs"]):
        traj, losses, converged = results_dict[label]
        c = COLOURS[idx % len(COLOURS)]
        ax.plot(traj[:, 0], traj[:, 1], color=c, alpha=0.85, linewidth=1.4, label=label)
        ax.scatter(traj[0, 0], traj[0, 1], color=c, marker="s", s=45, zorder=5)
        ax.scatter(
            traj[-1, 0],
            traj[-1, 1],
            color=c,
            marker="o",
            s=70,
            zorder=6,
            facecolors=c if converged else "none",
            linewidths=1.5,
        )
    ax.set_title(f"{problem['name']} — Optimiser Trajectories")
    ax.legend(fontsize=7.5, loc="upper left", ncol=2)


def make_individual_plots(problem, results_dict, axes):
    """One subplot per optimiser for detailed path inspection."""
    for idx, (label, _, _) in enumerate(problem["configs"]):
        ax = axes[idx // 3][idx % 3]
        _contour_base(problem, ax)
        traj, losses, converged = results_dict[label]
        c = COLOURS[idx % len(COLOURS)]
        ax.plot(traj[:, 0], traj[:, 1], color=c, alpha=0.85, linewidth=1.4)
        ax.scatter(traj[0, 0], traj[0, 1], color=c, marker="s", s=45, zorder=5)
        ax.scatter(
            traj[-1, 0],
            traj[-1, 1],
            color=c,
            marker="o",
            s=70,
            zorder=6,
            facecolors=c if converged else "none",
            linewidths=1.5,
        )
        ax.set_title(label, fontsize=10)

In [ ]:
# =====================================================================
# 6. Trajectory plots: combined overview + per-optimiser detail
# =====================================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))
for ax, problem in zip(axes, PROBLEMS):
    make_contour_plot(problem, results[problem["name"]], ax)
plt.tight_layout()
plt.show()

# Individual subplots for fine-grained inspection
for problem in PROBLEMS:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    make_individual_plots(problem, results[problem["name"]], axes)
    fig.suptitle(f"{problem['name']} — Individual Trajectories", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# =====================================================================
# 7. Loss convergence curves (log-scale for clarity)
# =====================================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, problem in zip(axes, PROBLEMS):
    name = problem["name"]
    for idx, (label, _, _) in enumerate(problem["configs"]):
        _, losses, _ = results[name][label]
        c = COLOURS[idx % len(COLOURS)]
        ax.plot(losses, color=c, linewidth=1.2, label=label)

    ax.axhline(y=1e-6, color="gray", linestyle=":", alpha=0.5, label="1e-6 threshold")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_yscale("log")
    ax.set_title(f"{name} — Loss vs Iteration")
    ax.legend(fontsize=8)
    ax.set_ylim(bottom=1e-10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Discussion

**Beale** (start = (-0.5, 0.0), 500 steps):
- **SGD**: Marches smoothly but slowly; lacks a mechanism to accelerate across flat regions and fails to reach the minimum within the step limit.
- **Momentum/NAG**: Perform best, utilizing inertia to rapidly cross the plateau. Both exhibit a characteristic, slight overshoot near the minimum.
- **AdaGrad**: Exhibits classic premature stopping. It stalls after a short descent, proving that its monotonically increasing cache rapidly kills the learning rate on unnormalized terrain.
- **RMSProp**: Converges successfully without high-frequency oscillations. Its trajectory consists of piecewise smooth segments joined by sharp turns, reflecting the somewhat rigid directional adjustments of pure second-moment adaptation.
- **Adam (lr=0.20)**: Converges efficiently, but the high initial learning rate causes it to sweep a wide arc off the optimal path before momentum corrects its course.

**Rosenbrock** (start = (-1.2, 1.0), 1000 steps):
- **SGD**: Demonstrates textbook zig-zagging. It bounces aggressively across the steep canyon walls, resulting in agonizingly slow progress along the valley floor.
- **Momentum/NAG**: Achieve the fastest convergence. Momentum suffers from structural bounces when initially entering the valley, whereas NAG carves a beautifully smooth path thanks to its look-ahead mechanism.
- **AdaGrad**: The massive initial gradient instantly explodes the cache, causing the optimizer to stall permanently after moving in a brief straight line.
- **RMSProp**: **Crucially, it does not zig-zag.** Instead, it traces a remarkably smooth arc along the valley floor. However, lacking a velocity mechanism, its progress is steady but highly conservative (slow) compared to Momentum and Adam.
- **Adam (lr=0.50)**: Traces a highly erratic, twisted path with self-intersections. This visually demonstrates how a high base LR combined with extreme gradient transitions (steep walls to flat valley) causes severe overcorrections and thrashing, even though its momentum eventually forces it to converge.

**Key takeaway:** Pure adaptive scaling (RMSProp) successfully suppresses perpendicular oscillations to yield a smooth path, but lacks the drive to accelerate along the valley floor. This visually confirms that conquering ill-conditioned landscapes efficiently requires combining adaptive step sizes with directional smoothing (e.g., properly tuned Adam or NAG).

In [ ]:
# =====================================================================
# 8. Summary table — how far from the minimum did each optimiser end up?
# =====================================================================

print(
    f"{'Optimizer':>10} | {'Beale dist':>10} | {'Beale loss':>10} | {'Rose dist':>10} | {'Rose loss':>10} |"
)
print("-" * 70)

for beale_label, _, _ in BEALE_CONFIG:
    # Look up results — both config lists share the same label names
    beale_traj, beale_losses, _ = results["Beale"][beale_label]
    rose_traj, rose_losses, _ = results["Rosenbrock"][beale_label]

    beale_dist = np.linalg.norm(beale_traj[-1] - np.array([3.0, 0.5]))
    rose_dist = np.linalg.norm(rose_traj[-1] - np.array([1.0, 1.0]))

    print(
        f"{beale_label:>10} | {beale_dist:>10.4f} | {beale_losses[-1]:>10.2e} |"
        f" {rose_dist:>10.4f} | {rose_losses[-1]:>10.2e} |"
    )

### Bonus: gradient normalisation (‖g‖ = 1)

We saw Adam needs a high LR because `1/√v̂` suppresses large initial gradients.
What if we strip all gradient magnitude before feeding it to the optimiser?

Below, every gradient is normalised to unit length before each step.
Can all optimisers now share the same LR?

In [ ]:
from core.nn import clip_grad_norm_

CLIP = 10.0  # generous threshold — prevents divergence, barely affects normal runs


def make_kw(clz, lr):
    if clz in (SGD, AdaGrad):
        return {"lr": lr}
    if clz in (Momentum, NAG):
        return {"lr": lr, "momentum": 0.9}
    if clz is RMSProp:
        return {"lr": lr, "beta": 0.9}
    return {"lr": lr, "betas": (0.9, 0.999)}


def run_raw(opt_class, kwargs, start, func, grad_fn, n_steps):
    """Run with gradient clipping to keep things finite for comparison."""
    param = Parameter(start.copy())
    opt = opt_class([param], **kwargs)
    for _ in range(n_steps):
        param.grad = grad_fn(param.data)
        clip_grad_norm_([param], CLIP)
        opt.step()
    return func(param.data)


def run_normalized(opt_class, kwargs, start, func, grad_fn, n_steps):
    """Same as run_raw but normalises ||g|| = 1 before every step."""
    param = Parameter(start.copy())
    opt = opt_class([param], **kwargs)
    for _ in range(n_steps):
        g = grad_fn(param.data)
        gn = np.linalg.norm(g)
        if gn > 1e-12:
            g = g / gn
        param.grad = g
        opt.step()
    return func(param.data)


TEST_LR = 0.01
ALL_OPTS = [
    ("SGD", SGD),
    ("Momentum", Momentum),
    ("NAG", NAG),
    ("AdaGrad", AdaGrad),
    ("RMSProp", RMSProp),
    ("Adam", Adam),
]

print(f"All optimisers with the SAME learning rate lr={TEST_LR}")
print(f"(raw runs use clip_grad_norm_(params, {CLIP}) to prevent divergence)")
print()
print(
    f"{'Optimiser':>10} | {'Beale (raw)':>12} | {'Beale (norm)':>12} |"
    f" {'Rose (raw)':>12} | {'Rose (norm)':>12} |"
)
print("-" * 70)

b_start = np.array([-0.5, 0.0])
r_start = np.array([-1.2, 1.0])

for label, clz in ALL_OPTS:
    kw = make_kw(clz, TEST_LR)
    loss_b_raw = run_raw(clz, kw, b_start, beale, grad_beale, 500)
    loss_b_norm = run_normalized(clz, kw, b_start, beale, grad_beale, 500)
    loss_r_raw = run_raw(clz, kw, r_start, rosenbrock, grad_rosenbrock, 1000)
    loss_r_norm = run_normalized(clz, kw, r_start, rosenbrock, grad_rosenbrock, 1000)
    print(
        f"  {label:>10} | {loss_b_raw:>12.2e} | {loss_b_norm:>12.2e} |"
        f" {loss_r_raw:>12.2e} | {loss_r_norm:>12.2e} |"
    )

### What normalisation reveals

**1. Gradient clipping is not a substitute for normalisation.**
Both raw and norm runs use `clip_grad_norm_(params, 10.0)`.  Clipping alone
prevents NaN divergence but doesn't equalise optimiser performance — the raw
Rosenbrock column still shows a wide spread of results across optimisers.

**2. Adam benefits most from normalisation.**
Without giant gradients to suppress, Adam's effective step matches the raw LR:
loss drops from 0.29 → 3.9e-5 (Beale) and 1.25 → 2.1e-4 (Rosenbrock).

**3. Does *not* unify LR requirements.**
Momentum and NAG barely improve — they rely on gradient magnitude to modulate
velocity.  AdaGrad also stays stuck (its cache accumulates from directional
changes, not magnitude).  Normalisation only removes the scale disparity;
each optimiser's internal dynamics still demand different LRs.

**Bottom line:** Gradient clipping is the safety net for pathological gradients
(Phase 4 2-2).  Gradient normalisation is a diagnostic tool that reveals how
much each optimiser depends on gradient magnitude.  Neither eliminates
per-optimiser tuning.